# CatBoost

## Introduction: model selection and boosting

Grid search and k-fold cross-validation are **model-selection and evaluation techniques**, not predictive models. XGBoost and CatBoost are **gradient-boosted decision-tree models**. This notebook trains CatBoost and evaluates it with 10-fold cross-validation; grid search and XGBoost are introduced here for context, not implemented below.

### Grid search: choosing hyperparameters

Hyperparameters control how a model learns, such as tree depth, learning rate, and the number of boosting iterations. Grid search evaluates every combination in a predefined set, usually using cross-validation, and selects the combination with the best mean validation score:

$\theta^* = \arg\max_{\theta \in \Theta} \frac{1}{K} \sum_{k=1}^{K} s_k(\theta)$

Here, $\Theta$ is the hyperparameter grid and $s_k$ is the validation score on fold $k$; for a loss, minimize instead. A grid with 3 depths and 4 learning rates contains 12 candidates, requiring 120 fits with 10-fold validation, plus an optional refit on all training data. Scikit-learn's `GridSearchCV` automates this process. Large grids are expensive, and repeatedly selecting on the same folds can overfit the validation results.

### K-fold cross-validation: estimating generalization

Split the training data into $K$ disjoint folds. For each fold, fit a fresh model on the other $K-1$ folds and evaluate it on the held-out fold, so each observation is used for validation once. Summarize the scores with their mean and standard deviation:

$\bar{s} = \frac{1}{K} \sum_{k=1}^{K} s_k, \qquad \sigma_s = \sqrt{\frac{1}{K} \sum_{k=1}^{K}(s_k-\bar{s})^2}$

The mean estimates performance across these splits; the standard deviation describes split-to-split variability, not a confidence interval. Below, `cross_val_score(..., cv=10)` uses accuracy and stratified folds by default for this classifier, preserving class proportions approximately without shuffling. Fit learned preprocessing inside each fold, ideally with a pipeline, to prevent leakage. Use group-aware or time-aware splits when observations are dependent.

### XGBoost: regularized gradient boosting

XGBoost (Extreme Gradient Boosting) builds an ensemble sequentially: each new tree aims to reduce the current ensemble's loss. For a raw prediction score $F_t(x)$, the update is

$F_t(x) = F_{t-1}(x) + \eta f_t(x)$

where $f_t$ is the new tree and $\eta$ is the learning rate. For binary classification, a sigmoid converts the raw score to a probability. XGBoost uses first- and second-order loss derivatives to choose tree updates and includes a tree-complexity penalty:

$\mathcal{L}^{(t)} = \sum_i \ell(y_i, F_{t-1}(x_i) + \eta f_t(x_i)) + \Omega(f_t)$

Important controls include `n_estimators`, `max_depth`, `learning_rate`, row/column subsampling, and regularization. It is effective on tabular data, but can overfit without tuning; early stopping uses a separate validation set to choose when to stop adding trees.

### CatBoost: the model used below

CatBoost (Categorical Boosting) is another gradient-boosted tree library, designed to handle categorical features when they are identified through `cat_features`. Its ordered target-statistic encodings use preceding observations in a permutation rather than an observation's own target, reducing target leakage. Its ordered boosting technique reduces prediction shift caused by reusing training targets during boosting; actual boosting mode depends on configuration. CatBoost commonly uses symmetric trees, with the same split rule at each level, for efficient prediction.

Useful hyperparameters include `iterations`, `depth`, `learning_rate`, and `l2_leaf_reg`. The code below uses `CatBoostClassifier()` defaults and does not specify categorical columns. Tree-based models generally do not need feature scaling, but correct feature types and leakage-free evaluation still matter.

### Putting the techniques together

Reserve the test set before any model selection. Use cross-validation on the training set to compare hyperparameters, refit the selected model on the full training set, then evaluate once on the untouched test set. For an unbiased cross-validation estimate of the entire tuning procedure, use nested cross-validation.

```mermaid
flowchart TD
    A[Dataset] --> B[Training set]
    A --> C[Untouched test set]
    B --> D[Grid search over CatBoost or XGBoost settings]
    D --> E[K-fold validation for each candidate]
    E --> F[Select best mean validation score]
    F --> G[Refit on full training set]
    G --> H[Final evaluation]
    C --> H
```

This diagram shows a recommended tuning workflow. The existing cells below demonstrate a simpler workflow: default CatBoost training, a test-set confusion matrix, and training-set cross-validation. Do not use the test-set results to guide later tuning.

## Importing the dataset

- for Google Colab, install CatBoost with `!pip install catboost` and upload the dataset to the Colab environment

In [4]:
import pandas as pd

dataset = pd.read_csv('../data/Data.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Splitting the dataset into the Training set and Test set

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## Training CatBoost on the Training set

In [6]:
from catboost import CatBoostClassifier

classifier = CatBoostClassifier()
classifier.fit(X_train, y_train)

Learning rate set to 0.007956
0:	learn: 0.6778277	total: 57.9ms	remaining: 57.8s
1:	learn: 0.6642869	total: 58.3ms	remaining: 29.1s
2:	learn: 0.6510583	total: 58.7ms	remaining: 19.5s
3:	learn: 0.6351684	total: 59.1ms	remaining: 14.7s
4:	learn: 0.6203906	total: 59.5ms	remaining: 11.8s
5:	learn: 0.6053562	total: 60ms	remaining: 9.93s
6:	learn: 0.5913363	total: 60.4ms	remaining: 8.56s
7:	learn: 0.5773888	total: 60.8ms	remaining: 7.54s
8:	learn: 0.5638395	total: 61.2ms	remaining: 6.74s
9:	learn: 0.5507421	total: 61.6ms	remaining: 6.1s
10:	learn: 0.5377200	total: 62ms	remaining: 5.57s
11:	learn: 0.5243873	total: 62.4ms	remaining: 5.14s
12:	learn: 0.5129033	total: 62.8ms	remaining: 4.76s
13:	learn: 0.5047205	total: 63.1ms	remaining: 4.45s
14:	learn: 0.4942406	total: 63.5ms	remaining: 4.17s
15:	learn: 0.4836253	total: 63.9ms	remaining: 3.93s
16:	learn: 0.4733355	total: 64.3ms	remaining: 3.72s
17:	learn: 0.4629417	total: 64.7ms	remaining: 3.53s
18:	learn: 0.4527778	total: 65.1ms	remaining: 3.3

CatBoostClassifier()

## Making the Confusion Matrix

In [7]:
from sklearn.metrics import confusion_matrix, accuracy_score

y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[84  3]
 [ 0 50]]


0.9781021897810219

## Applying k-Fold Cross Validation

In [8]:
from sklearn.model_selection import cross_val_score

accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=10)
print("Accuracy: {:.2f} %".format(accuracies.mean() * 100))
print("Standard Deviation: {:.2f} %".format(accuracies.std() * 100))

Learning rate set to 0.007604
0:	learn: 0.6772057	total: 543us	remaining: 543ms
1:	learn: 0.6652631	total: 917us	remaining: 458ms
2:	learn: 0.6511778	total: 1.24ms	remaining: 413ms
3:	learn: 0.6360097	total: 1.58ms	remaining: 394ms
4:	learn: 0.6221216	total: 1.92ms	remaining: 383ms
5:	learn: 0.6065690	total: 2.28ms	remaining: 378ms
6:	learn: 0.5931936	total: 2.63ms	remaining: 373ms
7:	learn: 0.5783827	total: 2.91ms	remaining: 361ms
8:	learn: 0.5646397	total: 3.24ms	remaining: 357ms
9:	learn: 0.5508871	total: 4.95ms	remaining: 490ms
10:	learn: 0.5390344	total: 5.31ms	remaining: 478ms
11:	learn: 0.5279564	total: 5.65ms	remaining: 465ms
12:	learn: 0.5181720	total: 6ms	remaining: 455ms
13:	learn: 0.5055411	total: 6.33ms	remaining: 446ms
14:	learn: 0.4943124	total: 6.65ms	remaining: 437ms
15:	learn: 0.4824867	total: 7.02ms	remaining: 432ms
16:	learn: 0.4729890	total: 7.35ms	remaining: 425ms
17:	learn: 0.4638985	total: 7.75ms	remaining: 423ms
18:	learn: 0.4549171	total: 8.01ms	remaining: 413